# ch9_10 · 표가 있는 Word 문서로 RAG 만들기

2026-10-15 수업의 `ch9_10` 화면을 따라 완성한 실습입니다. 기본 흐름은 **문서 읽기 → 표 구조 보존 → 분할 → Upstage 임베딩 → Pinecone 저장 → 검색 → 답변**입니다.

이 노트북은 `C:\ai\Source\07_LLM` 폴더에서 실행합니다. Jupyter 커널은 **Python (llm)** 을 선택하세요. API 요청은 실행할 때만 발생합니다. Pinecone 키는 파일에 저장하지 않고 실행 세션에만 입력합니다.

## 1. 실습 파일과 API 키 확인

`.env`의 설정을 읽습니다. 이 단계는 API를 호출하지 않으므로 키가 없어도 문서와 표를 확인할 수 있습니다. 4단계에서 필요한 키를 숨김 입력창으로 받을 수 있습니다. 입력한 키는 노트북 출력과 파일에 남기지 않습니다.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv

BASE_DIR = Path.cwd()
DOCX_PATH = BASE_DIR / "tax_docx" / "소득세법_with_table.docx"
assert DOCX_PATH.is_file(), f"파일을 찾지 못했습니다: {DOCX_PATH}"
load_dotenv(BASE_DIR / ".env", override=False)
print("문서:", DOCX_PATH.name)

## 2. 수업 방식: 텍스트만 읽었을 때 확인

`Docx2txtLoader`는 본문을 읽기 쉽지만 표의 열 관계가 흐려질 수 있습니다. 아래 코드는 비교용이며 Pinecone에는 올리지 않습니다.

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
flat_documents = Docx2txtLoader(str(DOCX_PATH)).load_and_split(text_splitter=splitter)
print("일반 텍스트 조각 수:", len(flat_documents))
print("세율표 주변 예시:\n", next((d.page_content[:900] for d in flat_documents if "과세표준" in d.page_content and "세율" in d.page_content), "찾지 못함"))

## 3. 개선: 표의 각 행과 열을 함께 보존

Word 파일의 문단과 표를 원래 순서대로 읽습니다. 본문은 1,500자씩 나누고, 세율표는 독립된 문서 한 조각으로 만듭니다. 표 헤더가 각 행에 붙으므로 검색 결과를 읽을 때 금액 구간과 세율의 관계가 남습니다.

In [ ]:
from docx import Document as WordDocument
from docx.table import Table
from langchain_core.documents import Document

word = WordDocument(DOCX_PATH)
source_name = DOCX_PATH.name
documents = []
paragraph_buffer = []
table_count = 0

def add_paragraphs():
    text = "\n".join(paragraph_buffer).strip()
    if text:
        for chunk in splitter.split_text(text):
            documents.append(Document(page_content=chunk, metadata={
                "source": source_name, "content_type": "paragraph"
            }))
    paragraph_buffer.clear()

for item in word.iter_inner_content():
    if isinstance(item, Table):
        add_paragraphs()
        table_count += 1
        rows = [[cell.text.replace("\n", " ").replace("|", "\\|").strip()
                 for cell in row.cells] for row in item.rows]
        if rows:
            header = rows[0]
            lines = [f"[표 {table_count}] {header[0]}",
                     " | ".join(header),
                     " | ".join(["---"] * len(header))]
            lines += [" | ".join(row) for row in rows[1:]]
            documents.append(Document(page_content="\n".join(lines), metadata={
                "source": source_name, "content_type": "table", "table_index": table_count
            }))
    elif item.text.strip():
        paragraph_buffer.append(item.text.strip())
add_paragraphs()

table_documents = [d for d in documents if d.metadata["content_type"] == "table"]
print("본문 조각:", len(documents) - len(table_documents), "| 표:", len(table_documents))
assert table_documents, "표가 추출되지 않았습니다."
assert "과세표준" in table_documents[0].page_content and "세율" in table_documents[0].page_content
print(table_documents[0].page_content)

## 4. Upstage 임베딩과 Pinecone 인덱스 만들기

`UpstageEmbeddings`에는 모델의 기본 이름 `solar-embedding-1-large`를 줍니다. 이 라이브러리가 문서에는 `-passage`, 질문에는 `-query`를 자동으로 붙입니다. 현재 설치된 라이브러리에서 `embedding-query`를 직접 넣으면 `-query-query`가 되어 오류가 나므로 수정했습니다.

인덱스가 이미 있으면 차원을 확인한 뒤 재사용합니다. 다른 차원의 인덱스는 덮어쓰지 않습니다.

In [ ]:
import time
from langchain_upstage import UpstageEmbeddings
from pinecone import Pinecone, ServerlessSpec
from openai import AuthenticationError

if not os.getenv("UPSTAGE_API_KEY"):
    os.environ["UPSTAGE_API_KEY"] = getpass("Upstage API 키 (이 커널에만 저장): ")
embedding = UpstageEmbeddings(model="solar-embedding-1-large")
try:
    dimension = len(embedding.embed_query("과세표준 확인"))
except AuthenticationError:
    os.environ["UPSTAGE_API_KEY"] = getpass("현재 Upstage 키가 무효입니다. 새 API 키 (이 커널에만 저장): ")
    embedding = UpstageEmbeddings(model="solar-embedding-1-large", api_key=os.environ["UPSTAGE_API_KEY"])
    dimension = len(embedding.embed_query("과세표준 확인"))
print("Upstage 임베딩 차원:", dimension)

INDEX_NAME = "tax-index-table"
NAMESPACE = "ch9-10-table-aware-v1"
if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Pinecone API 키 (이 커널에만 저장): ")
assert os.getenv("PINECONE_API_KEY"), "Pinecone API 키가 비어 있습니다."
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
existing = set(pc.list_indexes().names())
if INDEX_NAME not in existing:
    pc.create_index(name=INDEX_NAME, dimension=dimension, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    for _ in range(60):
        if pc.describe_index(INDEX_NAME).status["ready"]:
            break
        time.sleep(2)
    else:
        raise TimeoutError(f"{INDEX_NAME} 인덱스 준비 대기 시간 초과")
index_description = pc.describe_index(INDEX_NAME)
assert index_description.dimension == dimension, (
    f"기존 인덱스 차원 {index_description.dimension}과 임베딩 차원 {dimension}이 다릅니다."
)
print("Pinecone 인덱스 준비:", INDEX_NAME, "| 차원:", index_description.dimension)

## 5. 문서 저장

같은 문서 조각에는 같은 ID를 부여합니다. 이 셀을 다시 실행해도 같은 ID를 갱신하므로 중복이 쌓이지 않습니다. 수업용 별도 namespace를 사용합니다.

In [ ]:
import hashlib
from langchain_pinecone import PineconeVectorStore

store = PineconeVectorStore(index_name=INDEX_NAME, embedding=embedding,
                            namespace=NAMESPACE, pinecone_api_key=os.environ["PINECONE_API_KEY"])
ids = ["ch9-10-" + hashlib.sha256(
    (str(i) + "\n" + d.page_content).encode("utf-8")
).hexdigest()[:32] for i, d in enumerate(documents)]
written_ids = store.add_documents(documents, ids=ids)
print("저장한 조각 수:", len(written_ids), "| namespace:", NAMESPACE)
assert len(written_ids) == len(documents)

## 6. 검색 결과 확인

먼저 검색 결과의 원문과 출처를 확인합니다. 연봉 5천만 원이라는 정보만으로는 **과세표준**을 알 수 없으므로, 표의 경계를 묻는 질문으로 검색을 점검합니다.

In [ ]:
question = "종합소득 과세표준이 5,000만 원일 때 적용되는 세율 구간과 계산식은 무엇인가요?"
matches = store.similarity_search_with_score(question, k=4)
for rank, (doc, score) in enumerate(matches, 1):
    print(f"\n[{rank}] score={score:.4f} type={doc.metadata.get('content_type')} source={doc.metadata.get('source')}")
    print(doc.page_content[:900])
assert matches, "검색 결과가 없습니다."

## 7. 검색 문맥으로 답변하기

질문 → 검색 → 문맥 정리 → Upstage 모델 → 답변 순서입니다. 근거에 없는 금액은 계산하지 않도록 프롬프트에 명시했습니다. 학습용 예시이며 실제 세금 신고 판단에 사용하지 않습니다.

In [ ]:
from langchain_upstage import ChatUpstage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
def format_documents(retrieved):
    return "\n\n".join(
        f"[자료 {i} | {d.metadata.get('source')} | {d.metadata.get('content_type')}]\n{d.page_content}"
        for i, d in enumerate(retrieved, 1)
    )

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 제공된 소득세법 문서의 표를 설명하는 학습 도우미입니다. "
     "제공된 자료만 근거로 답하세요. 과세표준과 연봉을 구분하고, 필요한 공제 정보가 없으면 "
     "실제 세액을 계산할 수 없다고 말하세요. 자료가 부족하면 추측하지 마세요. "
     "답변 끝에 근거 자료 번호를 적으세요.\n\n자료:\n{context}"),
    ("human", "{query}"),
])
llm = ChatUpstage(model="solar-pro4", temperature=0)
rag_chain = ({"context": retriever | format_documents,
              "query": RunnablePassthrough()} | prompt | llm | StrOutputParser())
print(rag_chain.invoke(question))

## 8. 스스로 확인할 점

1. 위 검색 결과에 `content_type=table` 조각이 있는가?
2. 표의 `1,400만 원 초과 5,000만 원 이하`와 `5,000만 원 초과`가 구분되는가?
3. 질문을 `연봉 5천만 원의 실제 소득세는?`로 바꾸면, 추가 정보가 필요하다고 답하는가?

API 키는 커널 종료 시 메모리에서 사라집니다. 다음 실행 때 다시 입력하면 됩니다.